In [ ]:
%cd C:\Users\almei\Documents\GitHub\research_ner_leaderboard\

In [ ]:
from copy import deepcopy
from pathlib import Path
import yaml
from transformers import AutoTokenizer
from dotenv import load_dotenv
from langchain.agents import create_agent
from typing import Literal
from tqdm import tqdm
import pandas as pd
from pathlib import Path
import jsonlines
import json
from langchain_core.prompts import ChatPromptTemplate

from spesia_ner.datasets import ClinicalRecordsDataset
from spesia_ner.data_models import AgentAnnotationsList, Record

load_dotenv()

semantic_groups_to_consider = [
    'Anatomy',
    'Chemicals & Drugs',
    'Concepts & Ideas',
    'Devices',
    'Disorders',
    'Living Beings',
    'Organizations',
    'Phenomena',
    'Physiology',
    'Procedures'
]

train_dataset = ClinicalRecordsDataset(
    'data/SemClinBr/annotated_records', 
    label_type='semantic_groups', 
    semantic_groups_to_consider=semantic_groups_to_consider,
    split="train")

test_dataset = ClinicalRecordsDataset(
    'data/SemClinBr/annotated_records', 
    label_type='semantic_groups', 
    semantic_groups_to_consider=semantic_groups_to_consider,
    split="test")

prompts_path = Path('./prompts.yaml')

with open(prompts_path) as f:
    prompts = yaml.safe_load(f)

In [ ]:
prompt_data = {
    "entities": ", ".join(semantic_groups_to_consider),
    "example_text": train_dataset.records[0].text,
    "example_response": train_dataset.records[0].to_agent_annotations('semantic_groups').model_dump_json()
}

prompt_template = ChatPromptTemplate(prompts['extract_umls_semantic_groups'])

def format_prompt(prompt_template: list[dict[str, str]], prompt_data: list[dict[str, str]]) -> list[dict[str, str]]:
    prompt_template = deepcopy(prompt_template)
    for message in prompt_template:
        message['content'] = message['content'].format_map({k:v for k, v in prompt_data.items() if k in message['content']})
    return {'messages': prompt_template}

In [ ]:
def spans_overlap(a_start, a_end, b_start, b_end):
    # true if spans have any overlap (half-open [start,end) semantics)
    return (a_start < b_end) and (b_start < a_end)

def evaluate_record(pred_record: Record, 
                    target_record: Record, 
                    labels_to_consider: list[str],
                    label_type: Literal['tags', 'semantic_groups'],                    
                    ) -> dict[str, dict[str, int]]:
    
    # build per-label sets of (start,end,label)
    target = set()
    for ann in target_record.annotations:
        for label in getattr(ann, label_type):
            if label in labels_to_consider:
                target.add((ann.start, ann.end, label))
    preds = set()
    for ann in pred_record.annotations:
        for label in getattr(ann, label_type):
            if label in labels_to_consider:
                preds.add((ann.start, ann.end, label))

    # greedy one-to-one matching: for each pred find first overlapping target with same label
    matched_preds = set()
    matched_target = set()
    for p in preds:
        p_start, p_end, p_label = p
        for t in target:
            t_start, t_end, t_label = t
            if t_label == p_label and spans_overlap(p_start, p_end, t_start, t_end):
                matched_preds.add(p)
                matched_target.add(t)
                break

    # compute per-label counts
    metrics = {label: {"tp": 0, "fp": 0, "fn": 0} for label in labels_to_consider}
    # TPs
    for p in matched_preds:
        _, _, label = p
        metrics[label]["tp"] += 1
    # FPs: predicted items not matched
    for p in preds - matched_preds:
        _, _, label = p
        metrics[label]["fp"] += 1
    # FNs: target items not matched
    for t in target - matched_target:
        _, _, label = t
        metrics[label]["fn"] += 1

    return metrics

In [ ]:
from langchain.chat_models import init_chat_model

model_name = 'ollama:gpt-oss:20b'
dataset_name = 'SemClinBr'
reasoning_effort = None
batch_size = 1

model = init_chat_model(
    model_name,
    reasoning_effort=reasoning_effort,
    format=AgentAnnotationsList.model_json_schema(),
)

agent = create_agent(
    model=model,
    response_format=AgentAnnotationsList,  # Auto-selects ProviderStrategy
)

In [ ]:
from langchain.messages import AIMessage

# Predict labels with LLM
pred_records = []
target_records = test_dataset.records

for i in tqdm(range(0, len(target_records), batch_size), total=len(target_records)//batch_size):
    # prepare batch
    batch = test_dataset.records[i:i+batch_size]
    prompts = []
    for record in batch:
        prompt_data['input_text'] = record.text
        # prompts.append(format_prompt(prompt_template, prompt_data))
        prompts.append(prompt_template.invoke(prompt_data))
    # run inference
    if 'ollama' in model_name:
        batch_results = model.batch(prompts)
    else:
        batch_results = agent.batch(prompts)
    # convert to records
    for result, record in zip(batch_results, batch):
        if isinstance(result, AIMessage):
            agent_annotations = AgentAnnotationsList(**json.loads(batch_results[0].content))
            pred_record = agent_annotations.to_record(
                record.text,
                label_type='semantic_groups'
            )
        else:
            pred_record = result['structured_response'].to_record(
                record.text,
                label_type='semantic_groups'
            )

        pred_records.append(pred_record)
        target_record = record

In [ ]:
labels = semantic_groups_to_consider   # list of labels

# accumulate counts
totals = {lab: {'tp': 0, 'fp': 0, 'fn': 0} for lab in labels}
for p_rec, g_rec in zip(pred_records, target_records):
    rec_metrics = evaluate_record(p_rec, g_rec, labels, label_type='semantic_groups')
    for lab in labels:
        m = rec_metrics.get(lab, {'tp': 0, 'fp': 0, 'fn': 0})
        totals[lab]['tp'] += m['tp']
        totals[lab]['fp'] += m['fp']
        totals[lab]['fn'] += m['fn']

# compute per-label scores
rows = []
for lab in labels:
    tp = totals[lab]['tp']
    fp = totals[lab]['fp']
    fn = totals[lab]['fn']
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    rows.append({'label': lab, 'tp': tp, 'fp': fp, 'fn': fn,
                 'precision': precision, 'recall': recall, 'f1': f1})

df = pd.DataFrame(rows).set_index('label').sort_values('f1', ascending=False)
print(df)

# micro / macro summaries
micro_tp = sum(totals[l]['tp'] for l in labels)
micro_fp = sum(totals[l]['fp'] for l in labels)
micro_fn = sum(totals[l]['fn'] for l in labels)
micro_precision = micro_tp / (micro_tp + micro_fp) if (micro_tp + micro_fp) > 0 else 0.0
micro_recall = micro_tp / (micro_tp + micro_fn) if (micro_tp + micro_fn) > 0 else 0.0
micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if (micro_precision + micro_recall) > 0 else 0.0

macro_precision = df['precision'].mean()
macro_recall = df['recall'].mean()
macro_f1 = df['f1'].mean()

overall_metrics = [
    {
        'Metric type': 'Micro',
        'Precision': micro_precision,
        'Recall': micro_recall,
        'F1': micro_f1
    },
    {
        'Metric type': 'Macro',
        'Precision': macro_precision,
        'Recall': macro_recall,
        'F1': macro_f1
    }
]

overall_metrics_df = pd.DataFrame(overall_metrics)
print(overall_metrics_df)

In [ ]:
if reasoning_effort is not None:
    model_name += '-' + reasoning_effort

# relevant paths
predictions_base_path = Path('predictions')
model_predictions_path = predictions_base_path / model_name.split(':')[1]
model_predictions_path.mkdir(exist_ok=True, parents=True)

# save predictions
with jsonlines.open(model_predictions_path /'predictions.jsonl', mode='w') as writer:
    writer.write_all([json.loads(record.model_dump_json()) for record in pred_records])

# save metrics
df.to_csv(model_predictions_path / 'metrics.csv')

# save report
with open('report_template.md') as f:
    report_template = f.read()

with open(model_predictions_path / 'report.md', 'x') as f:
    f.write(report_template.format_map({
        'model_name': model_name,
        'dataset_name': dataset_name,
        'per_label_metrics': df.to_markdown(),
        'overall_metrics': overall_metrics_df.to_markdown(index=False)
    }))